In [ ]:
import parametric_pillows, importlib, visualization, wall_generation, numpy as np, inflation
importlib.reload(parametric_pillows);
importlib.reload(visualization);

length = 2.0
width = 0.2

from ipywidgets import interactive, widgets
def tubeForCurvature(kappa):
    visualization.plot_line_segments(*parametric_pillows.bentArc(length=length, width=width, curvature=kappa, numArcSegments=50), bbox=[[-1.75, 0.25], [-0.5, 2.25]])
iplot = interactive(tubeForCurvature, kappa = widgets.FloatSlider(min=0.01, max=np.pi, value=1.0, step=0.01))
iplot.children[-1].layout.height = '500px'
display(iplot)

In [ ]:
kappa = iplot.children[0].value
visualization.plot_line_segments(*parametric_pillows.bentArchChain(1, length=length, width=width, curvature=kappa, numArcSegments=50))

In [ ]:
kappa = iplot.children[0].value
pts, edges = parametric_pillows.bentArc(length=length, width=width, curvature=kappa, numArcSegments=20)
m, fuseMarkers = wall_generation.triangulate_channel_walls(*parametric_pillows.bentArchChain(4, length=length, width=width, curvature=kappa, numArcSegments=40), 0.0005)
# visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=6, height=6)

isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

import py_newton_optimizer
isheet.setUseTensionFieldEnergy(True)
niter = 5000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet.visualizationMesh(), width=1024, height=768)
viewer.showWireframe()
viewer.show()

In [ ]:
import time
isheet.pressure = 0.010
inflation.benchmark_reset()
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars, opts)
    viewer.update(False, isheet.visualizationMesh())
    if cr.numIters() < iterations_per_output: break
    time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
inflation.benchmark_report()